# 🎓 Решение: LangChain Tool Calling с Ollama (Локальная модель)

## 📋 Полная реализация для локальных моделей

Этот файл демонстрирует использование **локальных моделей через Ollama**.

## 📦 Установка зависимостей

**Требования:**
1. Установить Ollama: https://ollama.com/download
2. Скачать модель: `ollama pull llama3.2` (или другую)

In [1]:
!pip install langchain langchain-ollama langgraph yfinance requests -q

## 🔧 Импорт библиотек

In [1]:
import sys
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.prebuilt import create_react_agent
import yfinance as yf
import requests

## ✅ Решение 1: Получение цены на нефть

In [2]:
@tool
def get_oil_price() -> str:
    """
    Получить текущую цену на нефть (WTI Crude Oil) в долларах США.
    
    Returns:
        str: Текущая цена на нефть в формате "Цена нефти WTI: $XX.XX"
    """
    try:
        ticker = yf.Ticker("CL=F")
        data = ticker.history(period="1d")
        
        if data.empty:
            return "❌ Не удалось получить данные о цене на нефть (возможно, рынок закрыт)"
        
        price = data["Close"].iloc[-1]
        return f"Цена нефти WTI: ${price:.2f}"
        
    except Exception as e:
        return f"❌ Ошибка при получении цены на нефть: {str(e)}"


# Тест функции
print("Тест get_oil_price():")
print(get_oil_price.invoke({}))

Тест get_oil_price():


$CL=F: possibly delisted; no price data found  (period=1d)


❌ Не удалось получить данные о цене на нефть (возможно, рынок закрыт)


## ✅ Решение 2: Получение курса USD/KZT

In [3]:
@tool
def get_usd_to_kzt_rate() -> str:
    """
    Получить текущий курс обмена USD/KZT через Forex API.
    
    Returns:
        str: Курс обмена в формате "1 USD = XXX.XX KZT"
    """
    try:
        url = "https://api.exchangerate-api.com/v4/latest/USD"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        kzt_rate = data["rates"]["KZT"]
        return f"1 USD = {kzt_rate:.2f} KZT"
        
    except requests.exceptions.RequestException as e:
        return f"❌ Ошибка при запросе к API: {str(e)}"
    except KeyError:
        return "❌ Ошибка: курс KZT не найден в ответе API"
    except Exception as e:
        return f"❌ Ошибка при получении курса валют: {str(e)}"


# Тест функции
print("Тест get_usd_to_kzt_rate():")
print(get_usd_to_kzt_rate.invoke({}))

Тест get_usd_to_kzt_rate():
1 USD = 494.61 KZT


## ✅ Решение 3: Конвертация цены нефти в тенге

In [4]:
@tool
def convert_oil_price_to_kzt() -> str:
    """
    Конвертировать текущую цену на нефть из долларов в тенге.
    
    Returns:
        str: Подробная информация о цене нефти в USD и KZT
    """
    try:
        oil_ticker = yf.Ticker("CL=F")
        oil_data = oil_ticker.history(period="1d")
        
        if oil_data.empty:
            return "❌ Не удалось получить данные о цене на нефть (возможно, рынок закрыт)"
        
        oil_price_usd = oil_data["Close"].iloc[-1]
        
        url = "https://api.exchangerate-api.com/v4/latest/USD"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        kzt_rate = data["rates"]["KZT"]
        
        oil_price_kzt = oil_price_usd * kzt_rate
        
        return f"""
Цена на нефть WTI:
В долларах: ${oil_price_usd:.2f}
В тенге: {oil_price_kzt:.2f} KZT
Курс: 1 USD = {kzt_rate:.2f} KZT
"""
        
    except Exception as e:
        return f"❌ Ошибка при конвертации: {str(e)}"


# Тест функции
print("Тест convert_oil_price_to_kzt():")
print(convert_oil_price_to_kzt.invoke({}))

$CL=F: possibly delisted; no price data found  (period=1d)


Тест convert_oil_price_to_kzt():
❌ Не удалось получить данные о цене на нефть (возможно, рынок закрыт)


## 🤖 Создание LangChain агента с Ollama

### 📝 Вставьте название вашей модели:
Примеры моделей:
- `llama3.2` (рекомендуется)
- `llama3.1`
- `qwen2.5`
- `mistral`
- `gemma2`

**Важно:** Модель должна поддерживать tool calling (Llama 3.2+ рекомендуется)

In [ ]:

OLLAMA_MODEL = "llama3.2"  # Замените на вашу модель

# Инициализация локальной модели через Ollama
model = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0,
    # base_url="http://localhost:11434"  # Раскомментируйте если Ollama на другом порту
)

# Список инструментов
tools = [
    get_oil_price,
    get_usd_to_kzt_rate,
    convert_oil_price_to_kzt
]

# Создание агента с ReAct паттерном
agent = create_react_agent(model, tools)

print(f"\n✅ Агент создан с моделью: {OLLAMA_MODEL}")
print("📍 Ollama запущен локально (без интернета для модели)")

## 🧪 Тестирование агента

In [ ]:
# Тест 1: Цена на нефть
result = agent.invoke({"messages": [("user", "Какая сейчас цена на нефть?")]})
result["messages"][-1].content

In [ ]:
# Тест 2: Курс валют
result = agent.invoke({"messages": [("user", "Какой курс доллара к тенге?")]})
result["messages"][-1].content

In [ ]:
# Тест 3: Конвертация
result = agent.invoke({"messages": [("user", "Сколько стоит баррель нефти в тенге?")]})
result["messages"][-1].content

In [ ]:
# Тест 4: Комплексный запрос
result = agent.invoke({"messages": [("user", "Дай мне полную информацию о цене на нефть в долларах и тенге")]})
result["messages"][-1].content

## 🔎 Просмотр вызовов инструментов

Эта ячейка помогает увидеть, какие инструменты агент вызвал и что вернул каждый tool. 
Отредактируйте вопрос при необходимости и выполните код ниже.

In [ ]:
# 🔍 Просмотр вызовов инструментов и их ответов
debug_question = "Дай мне цену нефти и курс доллара"  # ← замените на свой вопрос
debug_result = agent.invoke({"messages": [("user", debug_question)]})

print(f"🧠 Вопрос: {debug_question}")
print(f"💬 Итоговый ответ: {debug_result['messages'][-1].content}")

print("
🔧 Журнал инструментов:")
found_tool_activity = False
for message in debug_result["messages"]:
    tool_calls = getattr(message, "tool_calls", None) or []
    if tool_calls:
        for call in tool_calls:
            found_tool_activity = True
            name = call.get("name", "неизвестно")
            args = call.get("args", {})
            print(f"
LLM вызвала инструмент: {name}")
            if args:
                print(f"Аргументы: {args}")
    if getattr(message, "type", "") == "tool":
        found_tool_activity = True
        tool_name = getattr(message, "name", "tool_result")
        content = message.content
        if isinstance(content, list):
            # LangChain иногда возвращает список частей сообщения
            content = "
".join(part.get("text", "") for part in content if isinstance(part, dict))
        print(f"↳ Ответ инструмента {tool_name}: {content}")

if not found_tool_activity:
    print("Инструменты не вызывались для этого запроса.")



## 🔍 Отличия от OpenAI версии

### Преимущества Ollama:
- ✅ **Полностью бесплатно** - нет API ключей
- ✅ **Работает офлайн** - модель локально
- ✅ **Приватность** - данные не уходят в облако
- ✅ **Нет лимитов** - используй сколько хочешь

### Компромиссы:
- ⚠️ **Скорость** - зависит от вашего железа
- ⚠️ **Качество** - зависит от модели (llama3.2 работает хорошо)
- ⚠️ **Требования** - нужен Ollama установлен

### Когда использовать:
| Сценарий | OpenAI | Ollama |
|---|---|---|
| Прототипирование | ✅ | ✅ |
| Продакшн | ✅ | ⚠️ |
| Обучение | ⚠️ (платно) | ✅ |
| Приватные данные | ❌ | ✅ |
| Без интернета | ❌ | ✅ |

## 💡 Дополнительно: Сравнение разных моделей

Попробуйте разные модели и сравните результаты:

In [ ]:
# Функция для тестирования разных моделей
def test_model(model_name: str, question: str):
    print(f"\n{'='*80}")
    print(f"Модель: {model_name}")
    print(f"{'='*80}")
    
    try:
        model = ChatOllama(model=model_name, temperature=0)
        agent = create_react_agent(model, tools)
        result = agent.invoke({"messages": [("user", question)]})
        print(result["messages"][-1].content)
    except Exception as e:
        print(f"❌ Ошибка: {e}")

# Раскомментируйте для тестирования разных моделей
# test_model("llama3.2", "Какая цена на нефть в тенге?")
# test_model("qwen2.5", "Какая цена на нефть в тенге?")
# test_model("mistral", "Какая цена на нефть в тенге?")

## 📚 Полезные команды Ollama

```bash
# Список доступных моделей
ollama list

# Скачать модель
ollama pull llama3.2

# Удалить модель
ollama rm llama3.2

# Запустить интерактивный чат
ollama run llama3.2

# Проверить статус Ollama
ollama ps
```